## Imports

In [22]:
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, normalize
import time
from scipy.stats import mode
%matplotlib inline

## Data

In [23]:
X = pd.read_csv("kmeans_data/data.csv", header=None).values 
y = pd.read_csv("kmeans_data/label.csv", header=None).values.ravel() 

In [24]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("unique labels:", np.unique(y))
print("label counts:", Counter(y))

X shape: (10000, 784)
y shape: (10000,)
unique labels: [0 1 2 3 4 5 6 7 8 9]
label counts: Counter({1: 1135, 2: 1032, 7: 1028, 3: 1010, 9: 1009, 4: 982, 0: 980, 8: 974, 6: 958, 5: 892})


## Preprocess

In [25]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

In [26]:
X_unit = normalize(X, norm='l2', axis=1)

In [27]:
X_nonneg = np.copy(X)
X_nonneg[X_nonneg < 0] = 0 

In [28]:
# K = number of clusters
K = len(np.unique(y))
print("K =", K)

K = 10


## Distance Functions

In [29]:
def euclidean_distances(X, centroids):
    # returns shape (n_samples, K)
    # uses broadcasting
    return np.sqrt(((X[:, None, :] - centroids[None, :, :])**2).sum(axis=2))

def cosine_dissimilarity(X, centroids):
    # assumes rows L2-normalized
    # cosine_similarity = dot product
    sim = np.dot(X, centroids.T)  
    return 1.0 - sim

def generalized_jaccard_dissimilarity(X, centroids):
    # assumes X and centroids are non-negative
    # J = sum(min) / sum(max)
    n, d = X.shape
    k = centroids.shape[0]
    dists = np.zeros((n, k))
    for j in range(k):
        a = X
        b = centroids[j]
        numer = np.minimum(a, b).sum(axis=1)
        denom = np.maximum(a, b).sum(axis=1) + 1e-12
        jacc = numer / denom
        dists[:, j] = 1.0 - jacc
    return dists

## Centroids

In [30]:
def init_centroids(X, K, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.choice(X.shape[0], size=K, replace=False)
    return X[idx].astype(float)

## Kmeans

In [31]:
def compute_SSE(X, centroids, labels, dissimilarity='euclidean'):
    # compute SSE using the corresponding dists; SSE uses squared Euclidean by default,
    # but for generality we compute sum of squared dissimilarities (consistent across metrics)
    if dissimilarity == 'euclidean':
        dists = euclidean_distances(X, centroids)
    elif dissimilarity == 'cosine':
        dists = cosine_dissimilarity(X, centroids)
    elif dissimilarity == 'jaccard':
        dists = generalized_jaccard_dissimilarity(X, centroids)
    chosen = dists[np.arange(len(labels)), labels]
    return (chosen**2).sum()

In [32]:
def kmeans(X, K, metric='euclidean', max_iters=500, tol=1e-6, seed=0, verbose=False):
    # pick appropriate X preprocessing expectation:
    assert metric in ('euclidean','cosine','jaccard')
    centroids = init_centroids(X, K, seed=seed)
    sse_history = []
    prev_centroids = centroids.copy()
    start = time.time()
    for it in range(1, max_iters+1):
        # compute distances
        if metric == 'euclidean':
            dists = euclidean_distances(X, centroids)
        elif metric == 'cosine':
            dists = cosine_dissimilarity(X, centroids)
        else:
            dists = generalized_jaccard_dissimilarity(X, centroids)
        labels = np.argmin(dists, axis=1)
        # compute SSE
        sse = (dists[np.arange(X.shape[0]), labels]**2).sum()
        sse_history.append(sse)
        # update centroids (mean of members)
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            members = X[labels == k]
            if len(members) == 0:
                # reinitialize empty cluster (random data point)
                new_centroids[k] = X[np.random.randint(0, X.shape[0])]
            else:
                new_centroids[k] = members.mean(axis=0)
        # stopping checks:
        centroid_shift = np.linalg.norm(new_centroids - centroids)
        if verbose: print(f"iter {it}: SSE={sse:.4f}, shift={centroid_shift:.6f}")
        # 1) no change in centroid position
        if centroid_shift <= tol:
            reason = "centroid_no_change"
            centroids = new_centroids
            break
        # 2) SSE increases compared to previous iteration
        if it > 1 and sse_history[-1] > sse_history[-2] + 1e-12:
            reason = "sse_increase"
            centroids = new_centroids
            break
        centroids = new_centroids
    else:
        reason = "max_iter"
    end = time.time()
    # final labels and sse recompute
    if metric == 'euclidean':
        dists = euclidean_distances(X, centroids)
    elif metric == 'cosine':
        dists = cosine_dissimilarity(X, centroids)
    else:
        dists = generalized_jaccard_dissimilarity(X, centroids)
    labels = np.argmin(dists, axis=1)
    final_sse = (dists[np.arange(X.shape[0]), labels]**2).sum()
    return dict(centroids=centroids, labels=labels, sse_history=sse_history,
                iterations=len(sse_history), time_taken=end-start, final_sse=final_sse, stop_reason=reason)

## Cluster Labelling

In [33]:
def cluster_majority_labels(labels, y_true, K):
    cluster_labels = np.zeros(K, dtype=int)
    for k in range(K):
        members = y_true[labels == k]
        if len(members) == 0:
            cluster_labels[k] = -1  # no members
        else:
            m = mode(members, keepdims=True)  # ensures result is array
            cluster_labels[k] = m.mode[0]
    return cluster_labels

In [34]:
def clustering_accuracy(labels, y_true, cluster_labels):
    pred = np.array([cluster_labels[l] if cluster_labels[l] != -1 else -1 for l in labels])
    valid = pred != -1
    acc = (pred[valid] == y_true[valid]).sum() / len(y_true[valid])
    return acc

## Results

In [35]:
results = {}

# Euclidean
res_euc = kmeans(X_std, K, metric='euclidean', max_iters=500, seed=0, verbose=False)
clabels = cluster_majority_labels(res_euc['labels'], y, K)
acc_euc = clustering_accuracy(res_euc['labels'], y, clabels)
res_euc.update({'accuracy': acc_euc})
results['euclidean'] = res_euc

# Cosine (use X_unit)
res_cos = kmeans(X_unit, K, metric='cosine', max_iters=500, seed=0, verbose=False)
clabels = cluster_majority_labels(res_cos['labels'], y, K)
acc_cos = clustering_accuracy(res_cos['labels'], y, clabels)
res_cos.update({'accuracy': acc_cos})
results['cosine'] = res_cos

# Jaccard (use X_nonneg)
res_jac = kmeans(X_nonneg, K, metric='jaccard', max_iters=500, seed=0, verbose=False)
clabels = cluster_majority_labels(res_jac['labels'], y, K)
acc_jac = clustering_accuracy(res_jac['labels'], y, clabels)
res_jac.update({'accuracy': acc_jac})
results['jaccard'] = res_jac

# summary
for name, r in results.items():
    print(name, "final_sse:", r['final_sse'], "accuracy:", r['accuracy'], "iters:", r['iterations'], "time(s):", r['time_taken'], "stop:", r['stop_reason'])

euclidean final_sse: 5555336.185024954 accuracy: 0.5224 iters: 111 time(s): 34.09946966171265 stop: centroid_no_change
cosine final_sse: 2110.4535549873153 accuracy: 0.4735 iters: 2 time(s): 0.09900403022766113 stop: sse_increase
jaccard final_sse: 3686.353147591522 accuracy: 0.5477 iters: 12 time(s): 5.354003190994263 stop: sse_increase


### Q1 Compare the SSEs of Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which method is better?

SSE measures the compactness of clusters — lower SSE values within the same distance metric indicate better clustering.
However, since Euclidean, Cosine, and Jaccard distances operate on different numerical scales, their raw SSE values cannot be compared directly.
Instead, the convergence behavior should be compared.
Among the three, Euclidean K-Means shows a smooth decrease in SSE and converges stably after 111 iterations, while Cosine and Jaccard versions terminate early due to SSE increases.
This indicates that Euclidean K-Means achieves more stable and consistent clustering performance for this dataset.

### Q2 Compare the accuracies of Euclidean-K-means Cosine-K-means, Jarcard-K-means. First, label each cluster using the majority vote label of the data points in that cluster. Later, compute the predictive accuracy of Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which metric is better?

To evaluate predictive accuracy, each cluster was labeled using the majority vote of its ground-truth labels, and the fraction of correctly labeled samples was computed.
Among the three methods, Jaccard K-Means achieved the highest predictive accuracy (≈ 54.8%), followed by Euclidean (≈ 52.2%) and Cosine (≈ 47.3%).
This suggests that the Jaccard-based similarity metric produced clusters that better matched the true underlying categories in this dataset.
While Euclidean distance gave more stable convergence, Jaccard provided slightly better alignment with actual labels, indicating that for this dataset (with non-negative, high-dimensional features), the overlap-based similarity measure captures class structure more effectively.

### Q3 Which method requires more iterations and times to converge?

When using the same stopping criteria (no centroid movement, SSE increase, or max iteration limit),
the Euclidean K-Means algorithm required the most iterations (111) and the longest time (~35 seconds) to converge.
However, it achieved a stable termination where centroids stopped changing.
The Cosine K-Means and Jaccard K-Means versions converged much faster (2 and 12 iterations respectively)
but both stopped early due to an increase in SSE, indicating unstable convergence or local oscillations.
Therefore, although Euclidean K-Means is the slowest, it is the most stable and reliably convergent method among the three.
Cosine and Jaccard are computationally cheaper but less robust for this dataset.


In [36]:
def kmeans_full(X, K, metric='euclidean', stop_type='centroid', max_iters=500, tol=1e-6, seed=0):
    centroids = init_centroids(X, K, seed)
    sse_history = []
    for it in range(1, max_iters+1):
        if metric == 'euclidean':
            dists = euclidean_distances(X, centroids)
        elif metric == 'cosine':
            dists = cosine_dissimilarity(X, centroids)
        else:
            dists = generalized_jaccard_dissimilarity(X, centroids)
        labels = np.argmin(dists, axis=1)
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            members = X[labels == k]
            new_centroids[k] = members.mean(axis=0) if len(members) else centroids[k]
        sse = (dists[np.arange(len(labels)), labels]**2).sum()
        sse_history.append(sse)
        shift = np.linalg.norm(new_centroids - centroids)

        # different stopping conditions
        if stop_type == 'centroid' and shift <= tol:
            break
        elif stop_type == 'sse_increase' and it > 1 and sse > sse_history[-2]:
            break
        elif stop_type == 'max_iter' and it == max_iters:
            break
        centroids = new_centroids
    return sse_history[-1], len(sse_history)


In [37]:
for metric in ['euclidean', 'cosine', 'jaccard']:
    for stop in ['centroid', 'sse_increase', 'max_iter']:
        sse, iters = kmeans_full(X_std if metric=='euclidean' else 
                                        X_unit if metric=='cosine' else X_nonneg,
                                        K, metric=metric, stop_type=stop)
        print(metric, stop, "final SSE:", sse, "iterations:", iters)


euclidean centroid final SSE: 5555336.185024954 iterations: 111
euclidean sse_increase final SSE: 5555336.185024954 iterations: 500
euclidean max_iter final SSE: 5555336.185024954 iterations: 500
cosine centroid final SSE: 1932.7169069690065 iterations: 43
cosine sse_increase final SSE: 2245.085466461329 iterations: 2
cosine max_iter final SSE: 1932.7169069690065 iterations: 500
jaccard centroid final SSE: 3690.8226221009477 iterations: 55
jaccard sse_increase final SSE: 3686.07575672872 iterations: 12
jaccard max_iter final SSE: 3690.8226221009477 iterations: 500


### Q4 Compare the SSEs of Euclidean-K-means Cosine-K-means, Jarcard-K-means with respect to the following three terminating conditions: when there is no change in centroid position, when the SSE value increases in the next iteration, when the maximum preset value (e.g., 100) of iteration is complete  
  
  
    
When applying each termination condition separately:  
- For Euclidean K-Means, all three criteria led to the same final SSE, confirming that the algorithm converged completely and consistently at iteration 111.  
- For Cosine K-Means, stopping due to an SSE increase after only 2 iterations produced a noticeably higher SSE, while the centroid and max-iteration runs both reached a lower, stable SSE.  
- For Jaccard K-Means, all three runs produced nearly identical SSE values, with small random variation.  
Overall, the centroid-based stop condition provided the most reliable convergence across all metrics, while the SSE-increase condition can terminate prematurely.
Among all methods, Euclidean K-Means remains the most stable and consistent in its convergence behavior.

### Q5 What are your summary observations or takeaways based on your algorithmic analysis?

Across all experiments:
- Euclidean K-Means showed the most stable convergence behavior. It required more iterations but consistently reached centroid stability with a smooth decline in SSE.
- Cosine K-Means converged in very few iterations but often stopped due to SSE increases, indicating sensitivity and less reliable optimization.
- Jaccard K-Means provided the highest predictive accuracy (~54–55 %), suggesting that for non-negative, image-like data, an overlap-based similarity metric can better reflect true class structure.
- When comparing termination rules, stopping based on centroid stability produced the most consistent and lowest SSE results, while stopping on SSE increase often caused premature termination.
  
Overall, Euclidean K-Means is the most stable and dependable algorithm for convergence, whereas Jaccard K-Means yields slightly better clustering accuracy for this specific dataset. Cosine K-Means is computationally faster but less stable and less accurate.

## Checks

In [39]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

kmeans_sklearn = KMeans(n_clusters=K, n_init=10, random_state=42)
labels_sklearn = kmeans_sklearn.fit_predict(X_scaled)

print("Sklearn inertia (SSE):", kmeans_sklearn.inertia_)

print("Final SSE:", res_euc['final_sse'])


Sklearn inertia (SSE): 5555473.331553124
Final SSE: 5555336.185024954
